# Convert the private source dataframe to trainable SemKey shards
Attach `thestonedape/task-aware-eeg2text-source`, use a high-RAM Kaggle session, and save `/kaggle/working/task-aware-eeg2text-sharded` as a new private dataset after every gate passes.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = 'e2b7f1b574328fe0e27d0279df6e536aee5c3bfa'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-aware-eeg2text-sharded'
assert REPO_URL.startswith('https://github.com/') and 'REPLACE_' not in REPO_URL
assert len(COMMIT) == 40 and all(c in '0123456789abcdef' for c in COMMIT.lower())

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Add a private Kaggle Secret named GITHUB_TOKEN with read access to the repository'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual == COMMIT

In [ ]:
sources = glob.glob('/kaggle/input/*/zuco_eeg_label_8variants.df')
checksums = glob.glob('/kaggle/input/*/source_checksums.json')
manifests = glob.glob('/kaggle/input/*/canonical_validation_manifest.csv')
assert len(sources) == len(checksums) == len(manifests) == 1, (sources, checksums, manifests)
provenance = json.load(open(checksums[0], encoding='utf-8'))
expected = provenance['files']['zuco_eeg_label_8variants.df']['sha256']
expected_manifest = provenance['files']['canonical_validation_manifest.csv']['sha256']
assert sha256(manifests[0]) == expected_manifest
print({'source': sources[0], 'expected_sha256': expected, 'manifest_sha256': expected_manifest})

In [ ]:
subprocess.run([sys.executable, os.path.join(WORKTREE, 'kaggle', 'prepare_shards.py'), '--dataframe', sources[0], '--output-root', OUTPUT, '--rows-per-shard', '128', '--expected-sha256', expected, '--expected-validation-manifest', manifests[0]], check=True)
os.makedirs(os.path.join(OUTPUT, 'manifests'), exist_ok=True)
shutil.copy2(manifests[0], os.path.join(OUTPUT, 'manifests', 'canonical_validation_manifest.csv'))
shutil.copy2(checksums[0], os.path.join(OUTPUT, 'metadata', 'source_checksums.json'))

In [ ]:
subprocess.run([sys.executable, os.path.join(WORKTREE, 'kaggle', 'smoke_input.py'), '--dataset-root', OUTPUT, '--batch-size', '1'], check=True)
subprocess.run([sys.executable, os.path.join(WORKTREE, 'kaggle', 'smoke_semkey_sharded_loader.py'), '--dataset-root', OUTPUT, '--phase', 'val'], check=True)
report = json.load(open(os.path.join(OUTPUT, 'metadata', 'canonical_full_contract_report.json'), encoding='utf-8'))
assert report['row_count'] == 22335 and report['frozen_validation_check']['status'] == 'pass', report
assert report['semkey_generated_labels']['status'] == 'not_fabricated'
print('SOURCE TO CANONICAL TRAINABLE SHARDS: PASS')